# Vectorial tight focusing with a 2D FFT: an implementation from first principles
**Research tutorial | angular-spectrum FFT | separate branch from the thesis integral method**

**Learning outcome:** Given any two complex-valued incident Jones components sampled at an objective pupil, calculate the three complex field components `Ex(x,y,z), Ey(x,y,z), Ez(x,y,z)` on a volume by performing **one 2D inverse FFT per axial plane**.

This is the Fourier-form Richards–Wolf *forward* solver; no inverse optimization or nonlinear optics is included. Derivation is linked to the lens matrix in Ferrer-García's PhD thesis, Eq. (2.2) and Richards–Wolf Eq. (2.3), and complements the five-integral decomposition in Eqs. (2.8)–(2.10).

**Key caveat:** A finite FFT grid produces a periodic, sampled Fourier approximation. A direct azimuthal Bessel integral can resolve a selected radial coordinate without such a transverse grid. We will compare the approaches physically and numerically.

**Scope:** monochromatic, homogeneous medium, ideal aplanatic objective, no aberrations or interfaces; length units are vacuum wavelengths. Numerical input is **pupil-plane Jones field**, not field at the focus.

## 1. The integral, the polarization rotation, and the angular spectrum

With `exp(-i omega t)` time dependence, `k=2*pi*n/lambda0`, `alpha=arcsin(NA/n)` and `rho_p = f*sin(theta)`, the ideal aplanatic Richards–Wolf equation is

$$\mathbf E(x,y,z)=-\frac{ikf}{2\pi}\int_0^{\alpha}\int_0^{2\pi}\sqrt{\cos\theta}\,\mathbf R(\theta,\phi)\begin{pmatrix}E_x^{\rm in}\\E_y^{\rm in}\end{pmatrix}e^{i(k_xx+k_yy+k_zz)}\sin\theta\,d\phi\,d\theta.$$

The ray polarization rotation is a **3 by 2** matrix (two input polarization degrees of freedom become three focal components), with first column
`[cos(theta)cos²(phi)+sin²(phi), (cos(theta)-1)sin(phi)cos(phi), -sin(theta)cos(phi)]`
and second column `[(cos(theta)-1)sin(phi)cos(phi), cos(theta)sin²(phi)+cos²(phi), -sin(theta)sin(phi)]`. 

**Derive the Jacobian yourself:** `kx=k sin(theta)cos(phi)`, `ky=k sin(theta)sin(phi)`, `kz=sqrt(k²-kx²-ky²)`. Then

$$dk_x\,dk_y=k^2\cos\theta\sin\theta\,d\theta\,d\phi.$$

Thus

$$\mathbf E(x,y,z)=\iint_{k_\perp\le k_0\mathrm{NA}}\mathbf A(k_x,k_y)e^{i(k_xx+k_yy+k_zz)}\,dk_x dk_y,$$

$$\boxed{\mathbf A(k_x,k_y)=-\frac{if}{2\pi k}\frac{\mathbf R(\theta,\phi)\mathbf E_{\rm in}(x_p,y_p)}{\sqrt{\cos\theta}}},\qquad x_p=f k_x/k,\quad y_p=f k_y/k.$$

**Subtlety:** `sqrt(cos(theta))` in the *angular* integral becomes `1/sqrt(cos(theta))` in the **Cartesian k-space density** because of the Jacobian. Do not use the angular integrand directly as the FFT input!

**Additional subtlety:** `k_perp <= k0*NA`, where `k0=2*pi/lambda0`; this is equivalent to `sin(theta)<=NA/n`, not `kx²+ky² <= (k*NA)²` when n!=1.

## 2. Coordinates and sampling: this decides whether the FFT is trustworthy

The FFT requires conjugate grids `Δkx=2*pi/(Nx*dx)` and `Δky=2*pi/(Ny*dy)`, and the transverse computational window is `Lx=Nx*dx`, `Ly=Ny*dy`. The highest representable Fourier frequency is approximately `pi/dx`; it must be **larger** than the objective cutoff `k0*NA`.

A smaller `dx` gives a larger k-space window but, for a fixed `N`, **coarser** k-grid sampling; increasing `N` at fixed `dx` improves k-grid sampling and the computational spatial window. This tradeoff is easy to miss.

For a centered output grid, use `fftshift(ifft2(ifftshift(A)))`. The numerical quadrature scale is `Nx*Ny*Δkx*Δky`. Missing this factor changes absolute amplitudes and breaks comparisons with the Debye integral. A more standard Fourier convention can absorb `(2*pi)^-2` into the spectrum; this notebook does not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from scipy.integrate import simpson
from scipy.special import jv

@dataclass
class Setup:
    lambda0: float = 1.0  # Vacuum wavelength; all lengths in lambda0 units
    n: float = 1.0
    NA: float = 0.85
    f: float = 1.0
    N: int = 128
    dx: float = 0.20
    @property
    def k0(self): return 2*np.pi/self.lambda0
    @property
    def k(self): return self.n*self.k0
    @property
    def alpha(self): return np.arcsin(self.NA/self.n)
    def validate(self):
        if not (0 < self.NA < self.n): raise ValueError("For this tutorial require 0 < NA < n.")
        if self.k0*self.NA >= np.pi/self.dx: raise ValueError("Aliasing: decrease dx.")
        if self.N%2: raise ValueError("Use even N to simplify centering.")
cfg=Setup(); cfg.validate()
x=(np.arange(cfg.N)-cfg.N//2)*cfg.dx
dk=2*np.pi/(cfg.N*cfg.dx)
kx=2*np.pi*np.fft.fftshift(np.fft.fftfreq(cfg.N,d=cfg.dx))
KX,KY=np.meshgrid(kx,kx,indexing="xy")
KR=np.hypot(KX,KY)
mask=KR<=cfg.k0*cfg.NA
KZ=np.sqrt(np.clip(cfg.k**2-KR**2,0,None))
ct=KZ/cfg.k
print("x window length =", cfg.N*cfg.dx, "Δk =", dk, "k cutoff =", cfg.k0*cfg.NA,
      "sampling points in pupil radius =", cfg.k0*cfg.NA/dk)
plt.figure(figsize=(4,4));plt.imshow(mask,origin="lower",extent=[kx[0],kx[-1],kx[0],kx[-1]])
plt.xlabel("kx");plt.ylabel("ky");plt.title("Objective NA support");plt.show()

## 3. Build physically meaningful incident fields
We write the input in pupil coordinates `Xp=f*kx/k`, `Yp=f*ky/k`. A pupil radius is `f*NA/n`. All input functions return **two complex arrays**, making it possible to represent arbitrary phase, polarization textures, measured fields or superpositions.

Example options: Gaussian x-polarized, an LG-like vortex, and a radially polarized pupil. At the center `rho=0`, a radial Jones vector is defined by continuity as zero (the azimuth is undefined there).

**Do not apply the `sqrt(cos(theta))` factor when constructing the pupil field**; it belongs to the lens operator.

In [ ]:
XP=cfg.f*KX/cfg.k; YP=cfg.f*KY/cfg.k
Rp=np.hypot(XP,YP)
Ph=np.arctan2(YP,XP)
pupil_radius=cfg.f*cfg.NA/cfg.n
q=Rp/pupil_radius

def gaussian_pupil(q):
    amplitude=np.exp(-(q/0.75)**2)*mask
    return amplitude.astype(complex),np.zeros_like(amplitude,dtype=complex)

def vortex_pupil(q,phi,charge=1):
    a=(q**abs(charge))*np.exp(-(q/0.75)**2)*np.exp(1j*charge*phi)*mask
    return a,0j*a

def radial_pupil(q,phi):
    a=q*np.exp(-(q/0.75)**2)*mask
    return a*np.cos(phi)+0j, a*np.sin(phi)+0j

Ex_in,Ey_in=gaussian_pupil(q)
assert Ex_in.shape==Ey_in.shape==(cfg.N,cfg.N)
assert not np.any(Ex_in[~mask])
print("Pupil energy proxy:",(np.abs(Ex_in)**2+np.abs(Ey_in)**2).sum()*dk**2)

## 4. The lens operator and an FFT plane
For `cos(theta)>0`, construct the 3D vector components with the full geometric rotation. Although only two pupil components are supplied, `Ez` will usually be nonzero at focus. We implement **exactly** the angular-spectrum density derived in §1.

**Do not multiply by apodization twice** and do not flip the sign of `Ez` accidentally. The Fourier `exp(+i*kz*z)` convention implies using `ifft2`, not `fft2`, for propagation with the convention specified.

Points outside the NA disk are explicitly set to zero, avoiding a potentially misleading `sqrt(kz)` outside the allowed propagating region.

In [ ]:
def angular_spectrum(ex,ey,cfg=cfg):
    if ex.shape!=(cfg.N,cfg.N) or ey.shape!=(cfg.N,cfg.N):
        raise ValueError("Both pupil Jones arrays must have shape (N,N).")
    c=ct
    # geometric polarization rotation (the columns are transverse to the ray)
    cps=np.zeros_like(c)
    sps=np.zeros_like(c)
    nonzero=KR>0
    cps[nonzero]=KX[nonzero]/KR[nonzero]
    sps[nonzero]=KY[nonzero]/KR[nonzero]
    cps[~nonzero]=1.0
    st=KR/cfg.k
    fex=(c*cps**2+sps**2)*ex+(c-1)*cps*sps*ey
    fey=(c-1)*cps*sps*ex+(c*sps**2+cps**2)*ey
    fez=-st*(cps*ex+sps*ey)
    apod=np.zeros_like(c)
    apod[mask]=1/np.sqrt(c[mask])
    const=-1j*cfg.f/(2*np.pi*cfg.k)
    return np.stack([const*fex*apod,const*fey*apod,const*fez*apod])
def focus_plane(ex,ey,z=0.,cfg=cfg):
    Ak=angular_spectrum(ex,ey,cfg)
    propagation=np.exp(1j*KZ*z)
    # ifft2 convention: positive exponent, divided by N*N
    E=np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(Ak*propagation,axes=(-2,-1)),
                                            axes=(-2,-1)),axes=(-2,-1))
    return E * (cfg.N**2*dk**2)

E0=focus_plane(Ex_in,Ey_in)
print("Focal components shape:",E0.shape,"max component modulus:",
      [float(np.max(np.abs(e))) for e in E0])

## 5. Plot the three complex components (not just total intensity)
Unlike a scalar paraxial Fourier transform, the objective generates a longitudinal field. Display both intensity and phase; zeros and phase winding cannot be inferred reliably from intensity plots alone.

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(12,7),constrained_layout=True)
for j,name in enumerate(("Ex","Ey","Ez")):
    axes[0,j].imshow(np.abs(E0[j])**2,origin="lower",extent=[x[0],x[-1],x[0],x[-1]])
    axes[0,j].set_title("|"+name+"|²")
    axes[1,j].imshow(np.angle(E0[j]),origin="lower",extent=[x[0],x[-1],x[0],x[-1]],
                    cmap="twilight",vmin=-np.pi,vmax=np.pi)
    axes[1,j].set_title("arg("+name+")")
plt.show()
print("Longitudinal intensity fraction:",np.sum(np.abs(E0[2])**2)/np.sum(np.abs(E0)**2))

## 6. A full 3D focal volume and the memory trap
One plane needs 3 complex arrays. A full `[Nz,3,Ny,Nx]` volume with `complex128` requires `Nz*3*N*N*16` bytes, excluding temporary FFT arrays. At `512³`, a 3-component volume alone exceeds 6 GB.

Prefer *streaming one plane at a time* if only XZ or selected slices are needed. Repeated 2D FFTs are convenient when axial positions are arbitrary. A 3D FFT/CZT can be worthwhile when sampling many planes on a suitable axial grid.

The function below allocates a modest demonstration volume; reduce `Nz` before running on a laptop.

In [ ]:
zvec=np.linspace(-1.6,1.6,17)
volume=np.stack([focus_plane(Ex_in,Ey_in,z) for z in zvec],axis=0)
print("Full volume [Nz,3,Ny,Nx] =",volume.shape,"bytes =",volume.nbytes)
fig,axs=plt.subplots(1,3,figsize=(12,3.5),constrained_layout=True)
yc=cfg.N//2
for j,name in enumerate(("Ex","Ey","Ez")):
    axs[j].imshow(np.abs(volume[:,j,yc,:])**2,origin="lower",
                  extent=[x[0],x[-1],zvec[0],zvec[-1]],aspect="auto")
    axs[j].set(xlabel="x / lambda0",ylabel="z / lambda0",title="XZ |"+name+"|²")
plt.show()

## 7. Independent verification I: Cartesian Debye sum at individual grid points
The FFT is a fast way of evaluating the **same discrete Fourier sum**. We check a few locations by direct evaluation of

$$\mathbf E(x_i,y_j,z)=\Delta k_x\Delta k_y\sum_{m,n}\mathbf A(k_{x,m},k_{y,n})e^{i(k_{x,m}x_i+k_{y,n}y_j+k_{z,mn}z)}.$$

For *discrete* equivalence the error should be near floating-point roundoff. This test catches most array-order, scaling, and sign mistakes but is **not** by itself a continuum quadrature accuracy test.

In [ ]:
ak=angular_spectrum(Ex_in,Ey_in)
ztest=.31; F=focus_plane(Ex_in,Ey_in,ztest)
for iy,ix in [(cfg.N//2,cfg.N//2),(cfg.N//2+2,cfg.N//2-3),(cfg.N//2-5,cfg.N//2+4)]:
    phase=np.exp(1j*(KX*x[ix]+KY*x[iy]+KZ*ztest))
    direct=np.sum(ak*phase,axis=(1,2))*dk**2
    actual=F[:,iy,ix]
    err=np.linalg.norm(actual-direct)/max(1e-30,np.linalg.norm(direct))
    print("point",iy,ix,"relative error",err)
    assert err<1e-10

## 8. Independent verification II: continuum Richards–Wolf quadrature at the origin
The previous test checks only algebra, not sampling accuracy. Now compare the FFT to a separate 2D `theta,phi` integration for a smooth Gaussian x-polarized pupil at `x=y=0`. Increase `N` and change `dx` independently to test convergence.

At `x=y=0`, azimuthal integration gives a particularly simple expression for the x component:

$$E_x(0,0,z)=-\frac{ikf}{2}\int_0^\alpha \sqrt{\cos\theta}\sin\theta\,(1+\cos\theta)\,h(\theta)e^{ikz\cos\theta}\,d\theta.$$

For our `gaussian_pupil`, `h(theta)=exp(-(sin(theta)/(0.75 sin(alpha)))²)`. The y and z components vanish on axis by symmetry (up to numerical error).

In [ ]:
t=np.linspace(0,cfg.alpha,2001)
h=np.exp(-(np.sin(t)/(0.75*np.sin(cfg.alpha)))**2)
def on_axis_reference(z):
    integral=simpson(np.sqrt(np.cos(t))*np.sin(t)*(1+np.cos(t))*
                     h*np.exp(1j*cfg.k*z*np.cos(t)),x=t)
    return (-1j*cfg.k*cfg.f/2)*integral
for z0 in (0.,.35,.8):
    value=focus_plane(Ex_in,Ey_in,z0)[:,cfg.N//2,cfg.N//2]
    ref=on_axis_reference(z0)
    rel=np.abs(value[0]-ref)/max(np.abs(ref),1e-30)
    print("z",z0,"Ex FFT",value[0],"continuum",ref,"relative difference",rel)
    assert np.isfinite(rel)
print("This result is a measured discrepancy, not an assumed accuracy threshold.")

## 9. Verification III: Maxwell transversality and spin-orbit conversion
At every pupil frequency, the angular spectrum must obey `kx*Ax+ky*Ay+kz*Az=0`. This is a direct check of the geometric lens transformation. The aperture and `sqrt(cos)` scaling do not spoil transversality.

Using radial polarization increases the longitudinal focal contribution; using `ell=1` vortex polarization makes spin-orbit structure apparent. These are **physical checks**, not just rendering tests.

In [ ]:
transverse=KX*ak[0]+KY*ak[1]+KZ*ak[2]
relative=np.linalg.norm(transverse[:, :])/max(1e-30,cfg.k*np.linalg.norm(ak))
print("k-dot-A normalized residual:",relative)
assert relative<1e-12
rx,ry=radial_pupil(q,Ph)
Erad=focus_plane(rx,ry)
frac=np.sum(np.abs(Erad[2])**2)/np.sum(np.abs(Erad)**2)
print("Radial-pupil focused longitudinal energy fraction in displayed plane:",frac)
vx,vy=vortex_pupil(q,Ph,charge=1)
Evort=focus_plane(vx,vy)
print("Charge +1 vortex at focus, component amplitudes at origin:",np.abs(Evort[:,cfg.N//2,cfg.N//2]))

## 10. How this differs from the thesis decomposition
The thesis starts with azimuthal LG factor `exp(i ell phi)`, uses the Jacobi–Anger identity to integrate `phi` analytically, and computes **five one-dimensional theta integrals** for each `rho,z`. This is efficient and precise for separable cylindrical modes and constitutes a strong independent numerical benchmark.

The FFT code starts with two arbitrary sampled complex pupil arrays and evaluates the **full transverse focal plane at once**. It makes no LG-mode assumption, but it replaces the angular integrals by sampling in Cartesian `kx,ky` and can suffer from aperture-edge and periodic-grid effects.

**Do not compare unnormalized arrays directly:** the FFT and integral notebook must use identical `lambda0,n,NA,f`, exactly the same input envelope `h` as a function of `sin(theta)/sin(alpha)`, and exactly the same Jones polarization and axial sign convention. The two computations also need distinct angular grids, so convergence should be checked *before* making accuracy claims.

## 11. Error checklist
- **Jacobian:** `sqrt(cos(theta))` becomes `1/sqrt(cos(theta))` in Cartesian angular-spectrum *density*.
- **Frequency cutoff:** `k_perp <= k0*NA`, and `NA/n <= 1`.
- **Sine-condition pupil mapping:** `xp=f*kx/k`; do not confuse pupil coordinates with focal coordinates.
- **Polarization rotation:** lens generates nonzero `Ez`; transversality `k·A=0` is mandatory.
- **Exponent sign:** `ifft2` realizes `exp(+i*kx*x)`; `exp(+i*kz*z)` means propagation toward positive z.
- **fftshift:** centered spectrum requires `ifftshift` before `ifft2`, `fftshift` after.
- **Normalization:** multiply `ifft2` by `N² Δkx Δky`; retain lens prefactor.
- **Phase:** never use `np.real` on complex fields prior to phase/topology analysis.
- **Aliasing:** `Δx < pi/(k0*NA)`, and pupil must have enough k-grid samples.
- **Sampling convergence:** test `N` and `dx` independently, not just zero-pad.
- **Finite FFT window:** the output is periodically continued; enlarge window for isolated field structures.
- **Volume memory:** stream axial planes when possible.
- **Validation:** direct discrete sum is an algebra check; independent continuum Debye integration is the physics check.
- **Limitations:** no medium interfaces, evanescent angular spectra, nonlinear interaction, or nonaplanatic aberration model in this tutorial.

**Exercises:** (1) replicate `ell=0` horizontal input using the 5-integral notebook; (2) repeat at `z != 0`; (3) plot `Δx,N` convergence; (4) implement sampling from a measured complex pupil Jones field; (5) benchmark time per plane against direct integration; (6) extend input to spatially varying elliptical polarization.